# 实验六 SmolVLM 网格定位与 RSI
使用实验六独立远端 venv 内核。先读 protocol.json 与 README.md；Notebook 不携带实验输出，已有结果时跳过运行单元。

In [ ]:
import os, json, subprocess
from pathlib import Path
ROOT = Path(os.environ.get('LAB06_ROOT', str(Path.home()/'llm_course_lab06')))
MODEL = Path(os.environ.get('LAB06_MODEL', str(Path.home()/'llm_course_lab05/models/SmolVLM-256M-Instruct')))
PY = ROOT/'.venv/bin/python'
CODE = ROOT/'src/hw6_rsi_localize.py'
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
def run(*args, gpu='0'):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=gpu)
    subprocess.run([str(PY),str(CODE),'--root',str(ROOT),'--model',str(MODEL),*args],env=env,check=True)
print(json.loads((ROOT/'src/protocol.json').read_text()))

## 1 数据与实现核验
55 张图、固定 seed=2026/42；种子 15 题与评测同题。已有数据不要再次生成。model_check 只反传检查，不做优化器更新。

In [ ]:
if not (ROOT/'hw6_imgs/gt.json').exists():
    run('--mode','make_data')
run('--mode','check')
run('--mode','model_check')

## 2 必做：4×4 三阶段与两轮 RSI
每阶段 10 步，FP32 + LoRA。此单元会训练；输出目录存在时拒绝覆盖。已有结果直接跳到第 4 节。

In [ ]:
run('--mode','run','--grid','4','--steps','10','--rounds','2',gpu='0')

## 3 选做 A：8×8 网格
使用同一组图、目标、初始化和优化步数；不修改 4×4 输出。

In [ ]:
run('--mode','run','--grid','8','--steps','10','--rounds','2',gpu='1')

## 4 结果核验与可视化
只重算派生汇总和图片，不训练；不会修改原始预测或训练日志。

In [ ]:
subprocess.run([str(PY),str(ROOT/'src/audit_results.py'),'--root',str(ROOT)],check=True)
from IPython.display import display, Image
for grid in [4,8]:
    out = ROOT/'outputs'/f'grid{grid}'
    print((out/'metrics.csv').read_text(encoding='utf-8-sig'))
    print((out/'round_metrics.csv').read_text(encoding='utf-8-sig'))
    display(Image(filename=str(out/'rsi_curves.png')))
    display(Image(filename=str(out/'training_loss.png')))
display(Image(filename=str(ROOT/'outputs/grid_comparison.png')))

## 5 查看原始纠错与数据回流
自查改写不等于改对；工具通过不等于 GT 正确；只有真实生成且通过双重检查的纠错答案才入池。某类代表样例未出现时，不编造。

In [ ]:
out = ROOT/'outputs/grid4'
for phase in ['sft_harness','rsi1_correction','rsi2_correction']:
    records = json.loads((out/phase/'records.json').read_text())
    for r in records:
        if not r['ok1']:
            print(r['id'],r['q'],r['pred1'],r['pred2_self'],r['pred2_tool'],r['final'],r['accepted'])
print(json.loads((out/'parameters.json').read_text()))